# <center style="font-weight: bold; color: #0098cd;">Transcripción automática del habla (ASR)</center>

## 1. Introducción

### 1.1 Objetivo

### 1.2 Contexto dentro del sistema completo

### 1.3 Requisitos de esta fase

## 2. Preparación del entorno de trabajo

### 2.1 Instalación de dependencias

In [76]:
# ============================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================
# Este notebook requiere la instalación previa de las dependencias del proyecto.
# Ejecutar en terminal:
# pip install -r requirements.txt

### 2.2 Importación de librerías

# PASAR ESTE CODIGO AL ESTILO DEL NOTEBOOK 1

In [135]:
# Gestión de rutas y sistema de archivos
from pathlib import Path
import soundfile as sf
import os
import json

# Manipulación de datos
import pandas as pd

# ASR (Whisper)
from faster_whisper import WhisperModel

# Evaluación
from jiwer import wer, cer, process_words

# Utilidades
from tqdm import tqdm

from collections import Counter

import re
import unicodedata
from num2words import num2words
import numpy as np

### 2.3 Gestión y configuración de rutas

In [115]:
# =========================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO
# =========================

project_root = Path.cwd()

while not (project_root / "data").exists():
    project_root = project_root.parent


# =========================
# DIRECTORIO PRINCIPAL DE DATOS
# =========================

data_dir = project_root / "data"


# =========================
# AUDIOS DE ENTRADA
# =========================

# Baseline (audios estandarizados)
standardized_audio_dir = data_dir / "standardized_audio"

# Pipeline final (audios procesados)
processed_audio_dir = data_dir / "processed_audio"


# =========================
# TRANSCRIPCIONES
# =========================

transcriptions_dir = data_dir / "transcriptions"

# Predicciones del ASR
predictions_dir = transcriptions_dir / "predictions"

# Baseline
pred_standardized_dir = predictions_dir / "standardized"

# Pipeline final
pred_processed_dir = predictions_dir / "processed"


# Ground truth (transcripciones manuales)
ground_truth_dir = transcriptions_dir / "ground_truth"


# =========================
# CREACIÓN DE DIRECTORIOS
# =========================

pred_standardized_dir.mkdir(parents=True, exist_ok=True)
pred_processed_dir.mkdir(parents=True, exist_ok=True)
ground_truth_dir.mkdir(parents=True, exist_ok=True)


# =========================
# VERIFICACIÓN
# =========================

print("Ruta raíz:", project_root)

print("\n--- Audios ---")
print("Baseline (standardized):", standardized_audio_dir)
print("Pipeline final (processed):", processed_audio_dir)

print("\n--- Transcripciones ---")
print("Predicciones baseline:", pred_standardized_dir)
print("Predicciones pipeline final:", pred_processed_dir)
print("Ground truth:", ground_truth_dir)

Ruta raíz: /Volumes/EXTENSION/GitHub/TFM

--- Audios ---
Baseline (standardized): /Volumes/EXTENSION/GitHub/TFM/data/standardized_audio
Pipeline final (processed): /Volumes/EXTENSION/GitHub/TFM/data/processed_audio

--- Transcripciones ---
Predicciones baseline: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions/standardized
Predicciones pipeline final: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions/processed
Ground truth: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/ground_truth


## 3. Preparación del *dataset* para ASR

### 3.1 Carga de audios preprocesados

In [116]:
# Carga de rutas de audios baseline (estandarizados)
audio_paths_standardized = sorted(list(standardized_audio_dir.glob("*.wav")))

# Carga de rutas de audios procesados (pipeline final)
audio_paths_processed = sorted(list(processed_audio_dir.glob("*.wav")))

# Verificación
print(f"Audios baseline (standardized): {len(audio_paths_standardized)}")
print(f"Audios pipeline final (processed): {len(audio_paths_processed)}")

Audios baseline (standardized): 11
Audios pipeline final (processed): 11


### 3.2 Asociación con metadatos

In [117]:
# Construcción de dataset baseline (audios estandarizados)
dataset_standardized = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_standardized
]

# Construcción de dataset procesado (pipeline final)
dataset_processed = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_processed
]

# Conversión a DataFrame para inspección
df_standardized = pd.DataFrame(dataset_standardized)
df_processed = pd.DataFrame(dataset_processed)

pd.DataFrame({
    "standardized": df_standardized["audio_id"].head(),
    "processed": df_processed["audio_id"].head()
})

,standardized,processed
0,AUDIO-2025-09-18-13-34-03,AUDIO-2025-09-18-13-34-03
1,AUDIO-2026-03-12-09-54-36,AUDIO-2026-03-12-09-54-36
2,AUDIO-2026-04-21-17-26-57,AUDIO-2026-04-21-17-26-57
3,AUDIO-2026-04-22-11-23-01,AUDIO-2026-04-22-11-23-01
4,AUDIO-2026-04-22-11-24-06,AUDIO-2026-04-22-11-24-06


### 3.3 Validación de entradas para transcripción

#### 3.3.1 Verificación de formato (wav, 16kHz, mono)

In [118]:
# Validación de audios baseline (estandarizados)
for item in dataset_standardized:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)


# Validación de audios procesados (pipeline final)
for item in dataset_processed:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)

#### 3.3.2 Control de errores y audios inválidos

In [119]:
# Filtrado baseline (audios estandarizados)
valid_standardized = [item for item in dataset_standardized if item["format_ok"]]
invalid_standardized = [item for item in dataset_standardized if not item["format_ok"]]

print(f"Standardized → válidos: {len(valid_standardized)}, inválidos: {len(invalid_standardized)}")


# Filtrado pipeline final (audios procesados)
valid_processed = [item for item in dataset_processed if item["format_ok"]]
invalid_processed = [item for item in dataset_processed if not item["format_ok"]]

print(f"Processed → válidos: {len(valid_processed)}, inválidos: {len(invalid_processed)}")


# Mostrar errores si existen
if invalid_standardized or invalid_processed:
    print("\nEjemplos de errores:")
    
    for item in (invalid_standardized + invalid_processed)[:5]:
        print(f"- {item['audio_id']}: {item.get('error', 'Formato incorrecto')}")

Standardized → válidos: 11, inválidos: 0
Processed → válidos: 11, inválidos: 0


## 4. Configuración del modelo ASR

### 4.1 Selección del modelo base (Whisper)

In [120]:
# Selección del tamaño del modelo
model_size = "medium"  # opciones: tiny, base, small, medium, large-v3

# Configuración de dispositivo (ajusta según tu equipo)
device = "cpu"   # "cuda" si tienes GPU
compute_type = "int8"  # "float16" en GPU, "int8" en CPU

# Carga del modelo
model = WhisperModel(model_size, device=device, compute_type=compute_type)

print(f"Modelo cargado: {model_size}")

Modelo cargado: medium


### 4.2 Parámetros de inferencia

In [121]:
asr_params = {
    "language": "es",                       # fuerza el idioma español, evitando autodetección y posibles errores de idioma
    "beam_size": 1,                         # número de hipótesis evaluadas; 1 = decodificación greedy (rápida y casi determinista)
    "condition_on_previous_text": False,    # evita usar contexto previo entre segmentos, reduciendo propagación de errores
    "word_timestamps": False,               # desactiva timestamps por palabra (innecesario si solo quieres texto)
    "temperature": 0.0                      # desactiva aleatoriedad para resultados consistentes
}

### 4.3 Definición de configuración *baseline*

In [122]:
# Configuración baseline utilizada en la transcripción

baseline_config = {
    "model_size": model_size,
    "device": device,
    "compute_type": compute_type,
    "asr_params": asr_params
}

print("Configuración baseline:")
display(baseline_config)

Configuración baseline:


{'model_size': 'medium',
 'device': 'cpu',
 'compute_type': 'int8',
 'asr_params': {'language': 'es',
  'beam_size': 1,
  'condition_on_previous_text': False,
  'word_timestamps': False,
  'temperature': 0.0}}

## 5. Transcripción automática del audio

INTRO AQUI

### 5.1 Generación de transcripciones (*baseline*)

In [123]:
# Transcripción baseline (audios estandarizados)
results_standardized = []

for item in tqdm(dataset_standardized):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_standardized.append({
        "audio_id": item["audio_id"],
        "text": text
    })


# Transcripción pipeline final (audios procesados)
results_processed = []

for item in tqdm(dataset_processed):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_processed.append({
        "audio_id": item["audio_id"],
        "text": text
    })

100%|██████████| 11/11 [02:50<00:00, 15.50s/it]


### 5.2 Almacenamiento estructurado de resultados

Antes de ejecutar este bloque, se recomienda limpiar manualmente los directorios de salida para evitar inconsistencias entre ejecuciones.

In [124]:
# Guardado baseline (audios estandarizados)
for item in results_standardized:
    output_path = pred_standardized_dir / f"{item['audio_id']}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)


# Guardado pipeline final (audios procesados)
for item in results_processed:
    output_path = pred_processed_dir / f"{item['audio_id']}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)

Las transcripciones generadas se almacenan en formato JSON, donde cada archivo representa un audio procesado. La estructura contiene el identificador del audio (`audio_id`) y el texto transcrito (`text`), facilitando su posterior uso en el cálculo de métricas y en las etapas de procesamiento del lenguaje natural.

Ejemplo:

```json
{
  "audio_id": "001",
  "text": "aplique fungicida en lote 3"
}
```

### 5.3 Verificación de transcripciones generadas

In [127]:
# Ejemplo de verificación

if results_standardized:
    print(results_standardized[0])

if results_processed:
    print(results_processed[0])

{'audio_id': 'AUDIO-2025-09-18-13-34-03', 'text': 'Buenas, José Manuel.  Mándame porfa si puedes algún teléfono de aquí de Ciuri o del encargado o del alcalde  o de alguien de aquí porfa.'}
{'audio_id': 'AUDIO-2025-09-18-13-34-03', 'text': 'Buenas, José Manuel.  Mándame porfa si puedes algún teléfono de aquí de Ciuri o del encargado o del alcalde  o de alguien de aquí porfa.'}


### 5.4 Control de errores

In [129]:
# Detección de textos vacíos en baseline
empty_standardized = [r for r in results_standardized if not r["text"].strip()]

# Detección de textos vacíos en pipeline final
empty_processed = [r for r in results_processed if not r["text"].strip()]

print(f"Baseline → textos vacíos: {len(empty_standardized)}")
print(f"Processed → textos vacíos: {len(empty_processed)}")

# Mostrar ejemplos si existen
if empty_standardized or empty_processed:
    print("\nEjemplos de audios con transcripción vacía:")
    
    for r in (empty_standardized + empty_processed)[:5]:
        print(f"- {r['audio_id']}")

Baseline → textos vacíos: 0
Processed → textos vacíos: 0


## 6. Evaluación del rendimiento del ASR

### 6.1 Definición del conjunto de referencia (*ground truth*)

subconjunto etiquetado manualmente

In [130]:
# Carga del archivo CSV de ground truth
df_gt = pd.read_csv(ground_truth_dir / "ground_truth.csv")

# Eliminamos posibles filas con valores nulos
df_gt = df_gt.dropna(subset=["audio_id", "transcripcion"])

# Verificamos que no existan identificadores duplicados
assert df_gt["audio_id"].is_unique, "Error: existen audio_id duplicados en el ground truth"

# Construcción del diccionario {audio_id: transcripcion}
ground_truth = dict(zip(df_gt["audio_id"], df_gt["transcripcion"]))

print(f"Total ground truth: {len(ground_truth)}")

Total ground truth: 11


### 6.2 Normalización previa de texto

esto se hace para...

In [131]:
# Función para normalizar texto antes de calcular métricas ASR
def normalize_text(text):
    
    # Conversión a minúsculas
    text = text.lower()
    
    # Conversión de números enteros a texto
    def replace_number(match):
        try:
            num = int(match.group())
            return num2words(num, lang='es')
        except:
            return match.group()
    
    text = re.sub(r"\b\d+\b", replace_number, text)
    
    # Eliminación de acentos
    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )
    
    # Eliminación de puntuación
    text = re.sub(r"[^\w\s]", "", text)
    
    # Normalización de espacios
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

### 6.3 Cálculo de métricas

INTRO AQUI

#### 6.3.1 *Word Error Rate* (WER)

In [132]:
# Función para calcular el Word Error Rate (WER) para un conjunto de resultados
def compute_wer(results, ground_truth):
    
    # Lista donde se almacenan los valores individuales de WER por audio
    scores = []
    
    # Iteramos sobre cada transcripción generada por el ASR
    for item in results:
        
        # Identificador único del audio
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en el conjunto ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo ASR
            hyp = item["text"]
            
            # Aplicamos la normalización para eliminar diferencias no relevantes
            ref = normalize_text(ref)
            hyp = normalize_text(hyp)
            
            # Evitamos casos vacíos tras normalización
            if ref and hyp:
                
                # Cálculo del WER entre referencia y predicción
                score = wer(ref, hyp)
                
                # Almacenamiento del resultado individual
                scores.append(score)
    
    # Devolvemos la lista completa de errores
    return scores

#### 6.3.2 *Character Error Rate* (CER)

In [133]:
# Función para calcular el Character Error Rate (CER) para un conjunto de resultados
def compute_cer(results, ground_truth):
    
    # Lista donde se almacenan los valores individuales de CER por audio
    scores = []
    
    # Iteramos sobre cada transcripción generada por el ASR
    for item in results:
        
        # Identificador único del audio
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en el conjunto ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo ASR
            hyp = item["text"]
            
            # Aplicamos la misma normalización que en WER para mantener coherencia
            ref = normalize_text(ref)
            hyp = normalize_text(hyp)
            
            # Evitamos casos vacíos tras normalización
            if ref and hyp:
                
                # Cálculo del CER entre referencia y predicción
                score = cer(ref, hyp)
                
                # Almacenamiento del resultado individual
                scores.append(score)
    
    # Devolvemos la lista completa de errores
    return scores

### 6.4 Evaluación cuantitativa

In [141]:
# Función para estadísticas robustas
def compute_robust_stats(values):
    arr = np.array(values)
    arr_sorted = np.sort(arr)
    n = len(arr_sorted)
    k = int(0.1 * n)
    trimmed = arr_sorted[k:n-k] if n > 2*k else arr_sorted
    
    return {
        "mean": np.mean(arr),
        "median": np.median(arr),
        "std": np.std(arr),
        "min": np.min(arr),
        "max": np.max(arr),
        "p90": np.percentile(arr, 90),
        "p95": np.percentile(arr, 95),
        "trimmed_mean": np.mean(trimmed)
    }


# Aplicación
stats_wer_standardized = compute_robust_stats(wer_standardized)
stats_wer_processed = compute_robust_stats(wer_processed)

stats_cer_standardized = compute_robust_stats(cer_standardized)
stats_cer_processed = compute_robust_stats(cer_processed)


# Conversión a DataFrame para visualización clara
df_stats = pd.DataFrame([
    {"metric": "WER", "pipeline": "standardized", **stats_wer_standardized},
    {"metric": "WER", "pipeline": "processed", **stats_wer_processed},
    {"metric": "CER", "pipeline": "standardized", **stats_cer_standardized},
    {"metric": "CER", "pipeline": "processed", **stats_cer_processed},
])

display(df_stats)

,metric,pipeline,mean,median,std,min,max,p90,p95,trimmed_mean
0,WER,standardized,0.062829,0.037037,0.053029,0.0,0.173913,0.117647,0.145780,0.057468
1,WER,processed,0.062835,0.033333,0.051178,0.0,0.173913,0.117647,0.145780,0.057474
2,CER,standardized,0.030851,0.017280,0.029235,0.0,0.088235,0.077551,0.082893,0.027903
3,CER,processed,0.030922,0.017380,0.027672,0.0,0.088235,0.077551,0.082893,0.027990


### 6.4 Análisis de resultados globales

In [170]:
# Comprobamos que los audio_id coinciden entre baseline y processed
ids_std = [r["audio_id"] for r in results_standardized]
ids_proc = [r["audio_id"] for r in results_processed]

assert ids_std == ids_proc, "Error: los audio_id no están alineados"


# Construcción de DataFrame con resultados individuales por audio
df_results = pd.DataFrame({
    "audio_id": ids_std,
    "wer_standardized": wer_standardized,
    "wer_processed": wer_processed,
    "cer_standardized": cer_standardized,
    "cer_processed": cer_processed
})


# Diferencia de error (clave para análisis real)
df_results["delta_wer"] = df_results["wer_processed"] - df_results["wer_standardized"]
df_results["delta_cer"] = df_results["cer_processed"] - df_results["cer_standardized"]


# Clasificación en tres estados: mejora / empeora / igual
df_results["wer_estado"] = df_results["delta_wer"].apply(
    lambda x: "mejora" if x < 0 else ("empeora" if x > 0 else "igual")
)

df_results["cer_estado"] = df_results["delta_cer"].apply(
    lambda x: "mejora" if x < 0 else ("empeora" if x > 0 else "igual")
)


# =========================
# RESUMEN GLOBAL (CLAVE)
# =========================

wer_summary = df_results["wer_estado"].value_counts(normalize=True) * 100
cer_summary = df_results["cer_estado"].value_counts(normalize=True) * 100

print("Distribución WER:")
for estado, valor in wer_summary.items():
    print(f"{estado}: {valor:.2f}%")

print("\nDistribución CER:")
for estado, valor in cer_summary.items():
    print(f"{estado}: {valor:.2f}%")


# =========================
# ANÁLISIS DE CASOS (WER)
# =========================

df_best_wer = df_results[df_results["wer_estado"] == "mejora"].sort_values(by="delta_wer")
df_worst_wer = df_results[df_results["wer_estado"] == "empeora"].sort_values(by="delta_wer", ascending=False)

print("\nTop mejoras (WER):")
display(df_best_wer)

print("\nPeores casos (WER):")
display(df_worst_wer)

# =========================
# ANÁLISIS DE CASOS (CER)
# =========================

df_best_cer = df_results[df_results["cer_estado"] == "mejora"].sort_values(by="delta_cer")
df_worst_cer = df_results[df_results["cer_estado"] == "empeora"].sort_values(by="delta_cer", ascending=False)

print("\nTop mejoras (CER):")
display(df_best_cer)

print("\nPeores casos (CER):")
display(df_worst_cer)


# =========================
# VISIÓN GENERAL
# =========================
print("\nVisión general:")
display(df_results)

Distribución WER:
igual: 54.55%
mejora: 27.27%
empeora: 18.18%

Distribución CER:
igual: 45.45%
mejora: 27.27%
empeora: 27.27%

Top mejoras (WER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
7,AUDIO-2026-04-25-10-36-43,0.099548,0.085973,0.054054,0.040075,-0.013575,-0.013979,mejora,mejora
2,AUDIO-2026-04-21-17-26-57,0.023077,0.015385,0.014265,0.011412,-0.007692,-0.002853,mejora,mejora
9,AUDIO-2026-04-25-10-37-58,0.037037,0.033333,0.017280,0.015778,-0.003704,-0.001503,mejora,mejora



Peores casos (WER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
4,AUDIO-2026-04-22-11-24-06,0.014388,0.028777,0.006684,0.017380,0.014388,0.010695,empeora,empeora
10,AUDIO-2026-04-25-10-38-08,0.010638,0.021277,0.007059,0.014118,0.010638,0.007059,empeora,empeora



Top mejoras (CER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
7,AUDIO-2026-04-25-10-36-43,0.099548,0.085973,0.054054,0.040075,-0.013575,-0.013979,mejora,mejora
2,AUDIO-2026-04-21-17-26-57,0.023077,0.015385,0.014265,0.011412,-0.007692,-0.002853,mejora,mejora
9,AUDIO-2026-04-25-10-37-58,0.037037,0.033333,0.017280,0.015778,-0.003704,-0.001503,mejora,mejora



Peores casos (CER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
4,AUDIO-2026-04-22-11-24-06,0.014388,0.028777,0.006684,0.017380,0.014388,0.010695,empeora,empeora
10,AUDIO-2026-04-25-10-38-08,0.010638,0.021277,0.007059,0.014118,0.010638,0.007059,empeora,empeora
6,AUDIO-2026-04-25-10-36-13,0.095890,0.095890,0.036685,0.038043,0.000000,0.001359,igual,empeora



Visión general:


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
0,AUDIO-2025-09-18-13-34-03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,igual,igual
1,AUDIO-2026-03-12-09-54-36,0.117647,0.117647,0.077551,0.077551,0.000000,0.000000,igual,igual
2,AUDIO-2026-04-21-17-26-57,0.023077,0.015385,0.014265,0.011412,-0.007692,-0.002853,mejora,mejora
3,AUDIO-2026-04-22-11-23-01,0.173913,0.173913,0.088235,0.088235,0.000000,0.000000,igual,igual
4,AUDIO-2026-04-22-11-24-06,0.014388,0.028777,0.006684,0.017380,0.014388,0.010695,empeora,empeora
5,AUDIO-2026-04-25-10-35-47,0.028986,0.028986,0.002928,0.002928,0.000000,0.000000,igual,igual
6,AUDIO-2026-04-25-10-36-13,0.095890,0.095890,0.036685,0.038043,0.000000,0.001359,igual,empeora
7,AUDIO-2026-04-25-10-36-43,0.099548,0.085973,0.054054,0.040075,-0.013575,-0.013979,mejora,mejora
8,AUDIO-2026-04-25-10-37-29,0.090000,0.090000,0.034623,0.034623,0.000000,0.000000,igual,igual
9,AUDIO-2026-04-25-10-37-58,0.037037,0.033333,0.017280,0.015778,-0.003704,-0.001503,mejora,mejora


## 7. Análisis de errores de transcripción

### 7.1 Identificación de errores frecuentes

#### 7.1.1 Sustituciones

In [178]:
# Función para obtener las sustituciones más frecuentes
def get_substitutions(results, ground_truth):
    
    subs = Counter()
    
    for item in results:
        audio_id = item["audio_id"]
        
        if audio_id in ground_truth:
            
            # Normalización coherente con WER/CER
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                ref_words = ref.split()
                hyp_words = hyp.split()
                
                alignment = process_words(ref, hyp)
                
                for group in alignment.alignments:
                    for chunk in group:
                        
                        if chunk.type == "substitute":
                            
                            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
                            
                            # FILTRO CLAVE: solo sustituciones 1 a 1 (palabra a palabra)
                            if len(ref_segment) == 1 and len(hyp_segment) == 1:
                                
                                ref_word = ref_segment[0]
                                hyp_word = hyp_segment[0]
                                
                                subs[(ref_word, hyp_word)] += 1
    
    return subs


# Cálculo
subs_standardized = get_substitutions(results_standardized, ground_truth)
subs_processed = get_substitutions(results_processed, ground_truth)


# Visualización
print("Sustituciones más frecuentes (filtradas):")

print("\nStandardized:")
for (ref, hyp), count in subs_standardized.most_common(10):
    print(f"{ref} → {hyp} ({count})")

print("\nProcessed:")
for (ref, hyp), count in subs_processed.most_common(10):
    print(f"{ref} → {hyp} ({count})")

Sustituciones más frecuentes (filtradas):

Standardized:
miga → amiga (2)
llamado → damos (1)
yo → ya (1)
alejarme → dejarme (1)
turrialba → rurea (1)
taladro → taladron (1)
chulo → chuno (1)
medio → mediodia (1)
logrono → grono (1)
he → estaba (1)

Processed:
llamado → damos (1)
yo → ya (1)
alejarme → dejarme (1)
turrialba → rugula (1)
taladro → taladron (1)
medio → mediodia (1)
logrono → grono (1)
he → estaba (1)
tiquetes → etiquetes (1)
te → si (1)


#### 7.1.2 Inserciones

In [179]:
# Función para calcular las inserciones más frecuentes
def get_insertions(results, ground_truth):
    
    ins = Counter()
    
    for item in results:
        audio_id = item["audio_id"]
        
        if audio_id in ground_truth:
            
            # Normalización coherente
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                hyp_words = hyp.split()
                
                alignment = process_words(ref, hyp)
                
                for group in alignment.alignments:
                    for chunk in group:
                        
                        if chunk.type == "insert":
                            
                            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
                            
                            # FILTRO: solo palabras individuales
                            if len(hyp_segment) == 1:
                                
                                hyp_word = hyp_segment[0]
                                
                                ins[hyp_word] += 1
    
    return ins


# Cálculo
ins_standardized = get_insertions(results_standardized, ground_truth)
ins_processed = get_insertions(results_processed, ground_truth)


# Visualización
print("Inserciones más frecuentes (filtradas):")

print("\nStandardized:")
for word, count in ins_standardized.most_common(10):
    print(f"{word} ({count})")

print("\nProcessed:")
for word, count in ins_processed.most_common(10):
    print(f"{word} ({count})")

Inserciones más frecuentes (filtradas):

Standardized:
y (1)
lo (1)
no (1)
a (1)
tenia (1)
rey (1)
de (1)
logrona (1)
que (1)

Processed:
hay (1)
lo (1)
no (1)
si (1)
de (1)
y (1)
ha (1)


#### 7.1.3 Eliminaciones

In [180]:
# Función para calcular las eliminaciones más frecuentes
def get_deletions(results, ground_truth):
    
    dels = Counter()
    
    for item in results:
        audio_id = item["audio_id"]
        
        if audio_id in ground_truth:
            
            # Normalización coherente
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                ref_words = ref.split()
                
                alignment = process_words(ref, hyp)
                
                for group in alignment.alignments:
                    for chunk in group:
                        
                        if chunk.type == "delete":
                            
                            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                            
                            # FILTRO: solo palabras individuales
                            if len(ref_segment) == 1:
                                
                                ref_word = ref_segment[0]
                                
                                dels[ref_word] += 1
    
    return dels


# Cálculo
dels_standardized = get_deletions(results_standardized, ground_truth)
dels_processed = get_deletions(results_processed, ground_truth)


# Visualización
print("Eliminaciones más frecuentes (filtradas):")

print("\nStandardized:")
for word, count in dels_standardized.most_common(10):
    print(f"{word} ({count})")

print("\nProcessed:")
for word, count in dels_processed.most_common(10):
    print(f"{word} ({count})")

Eliminaciones más frecuentes (filtradas):

Standardized:
vale (2)
hora (1)
hay (1)
dia (1)
estado (1)
sea (1)
audio (1)
bua (1)
me (1)
ay (1)

Processed:
vale (2)
hora (1)
hay (1)
dia (1)
estado (1)
yo (1)
whore (1)
el (1)
audio (1)
bua (1)


#### 7.1.4 Resumen cuantitativo de errores

In [184]:
# Construcción de tabla resumen
df_error_summary = pd.DataFrame({
    "Pipeline": ["Standardized", "Processed"],
    "Sustituciones": [
        sum(subs_standardized.values()),
        sum(subs_processed.values())
    ],
    "Inserciones": [
        sum(ins_standardized.values()),
        sum(ins_processed.values())
    ],
    "Eliminaciones": [
        sum(dels_standardized.values()),
        sum(dels_processed.values())
    ]
})

# Total
df_error_summary["Total"] = (
    df_error_summary["Sustituciones"] +
    df_error_summary["Inserciones"] +
    df_error_summary["Eliminaciones"]
)

display(df_error_summary)

,Pipeline,Sustituciones,Inserciones,Eliminaciones,Total
0,Standardized,28,9,15,52
1,Processed,27,7,15,49


#### 7.1.5 Conclusiones (completar nombre con nombre de seccion)

👉 qué tipo de errores predominan

👉 si cambian entre pipelines

👉 impacto real del preprocesado

### 7.2 Análisis de errores con impacto en la estructuración

#### 7.2.1 Términos de dominio

Buenos días,

Para poder avanzar en el análisis del impacto de los errores de transcripción en la estructuración de la información, necesito definir un conjunto de términos de dominio agrícola relevantes para el proyecto.

Os agradecería que me proporcionaseis un listado estructurado de términos, organizado por categorías, que incluya:

- Cultivos (ej: café, maíz, trigo…)
- Variedades de cultivo (si aplica)
- Plagas y enfermedades (ej: roya, gusano…)
- Productos fitosanitarios (fungicidas, herbicidas…)
- Prácticas agrícolas (ej: siembra, riego, fumigación…)
- Estados del cultivo (ej: crecimiento, floración, maduración…)
- Unidades de parcela (lote, finca, parcela…)
- Condiciones climáticas (lluvia, sequía, humedad…)
- Herramientas o maquinaria (tractor, bomba, etc.) [opcional]
- Unidades de medida (kilos, hectáreas, litros…)
- Cantidades y expresiones numéricas habituales

Para cada término, es importante incluir también:

* Sinónimos habituales
* Formas coloquiales
* Variantes sin acento
* Posibles formas en las que los agricultores suelen referirse a ellos

Si es posible, el formato ideal sería un JSON estructurado por categorías.

Este listado se utilizará exclusivamente para analizar cómo los errores del ASR afectan a la extracción de información relevante del dominio.

Gracias de antemano.

## ✔ Ejemplo de JSON (lo que deberían darte)

```json
{
  "cultivo_cafe": ["café", "cafetal"],
  "plaga": ["plaga", "bicho", "insecto"],
  "producto_fitosanitario": ["fungicida", "herbicida", "insecticida", "veneno"],
  "unidad_parcela": ["lote", "parcela", "finca"],
  "cosecha": ["cosecha", "recolección"]
}
```

## ✔ Cómo lo justificas en el TFM

> Los términos de dominio son definidos por expertos y almacenados en un archivo externo en formato JSON, lo que permite su actualización independiente del código y facilita su reutilización en etapas posteriores del pipeline.

In [ ]:
# Definición de términos de dominio relevantes para el análisis
domain_terms = ["café", "plaga", "fungicida", "lote", "cosecha"]

# Función para calcular errores en términos de dominio
def domain_error_count(results, ground_truth):
    count = 0

    for item in results:
        audio_id = item["audio_id"]

        # Verificamos que exista referencia en ground truth
        if audio_id in ground_truth:

            # Normalizamos texto a minúsculas para comparación robusta
            ref = ground_truth[audio_id].lower()
            hyp = item["text"].lower()

            # Comprobamos pérdida de términos de dominio
            for term in domain_terms:
                if term in ref and term not in hyp:
                    count += 1

    return count

#### 7.2.2 Números y cantidades

In [ ]:
# Función para extraer números en formato dígito y texto
def extract_numbers(text):
    return re.findall(
        r"\d+[.,]?\d*|\b(cero|uno|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|"
        r"diez|once|doce|trece|catorce|quince|veinte|treinta|cuarenta|"
        r"cincuenta|sesenta|setenta|ochenta|noventa|cien|ciento|mil)\b",
        text.lower()
    )

# Función para calcular errores en números
def number_error_count(results, ground_truth):
    count = 0

    for item in results:
        audio_id = item["audio_id"]

        # Verificamos existencia en ground truth
        if audio_id in ground_truth:

            # Extracción de números
            ref_nums = extract_numbers(ground_truth[audio_id])
            hyp_nums = extract_numbers(item["text"])

            # Comprobamos discrepancias
            if ref_nums != hyp_nums:
                count += 1

    return count

#### 7.2.3 Resumen cuantitativo del impacto

In [ ]:
# Cálculo de errores para cada pipeline
domain_std = domain_error_count(results_standardized, ground_truth)
domain_no_vad = domain_error_count(results_without_vad, ground_truth)
domain_vad = domain_error_count(results_conditional_vad, ground_truth)

name_std = name_error_count(results_standardized, ground_truth)
name_no_vad = name_error_count(results_without_vad, ground_truth)
name_vad = name_error_count(results_conditional_vad, ground_truth)

number_std = number_error_count(results_standardized, ground_truth)
number_no_vad = number_error_count(results_without_vad, ground_truth)
number_vad = number_error_count(results_conditional_vad, ground_truth)


# Construcción de tabla resumen
import pandas as pd

df_struct_errors = pd.DataFrame({
    "Pipeline": ["Standardized", "Sin VAD", "VAD condicional"],
    "Errores dominio": [domain_std, domain_no_vad, domain_vad],
    "Errores nombres": [name_std, name_no_vad, name_vad],
    "Errores números": [number_std, number_no_vad, number_vad]
})

# Cálculo del impacto total
df_struct_errors["Total impacto"] = (
    df_struct_errors["Errores dominio"] +
    df_struct_errors["Errores nombres"] +
    df_struct_errors["Errores números"]
)

display(df_struct_errors)

#### 7.2.4 Conclusiones (completar con nombre del subapartado)

### 7.3 Impacto de los errores en etapas posteriores (PLN)

Los errores de transcripción identificados tienen un impacto directo en las etapas posteriores del pipeline de PLN. En particular, las sustituciones que afectan a términos de dominio pueden provocar la pérdida de entidades relevantes, dificultando su extracción y posterior estructuración. Asimismo, errores en verbos clave pueden alterar la interpretación del mensaje, afectando a la clasificación del tipo de entrada. Estos efectos evidencian que la calidad del ASR condiciona directamente el rendimiento de las etapas de procesamiento del lenguaje natural.

## 8. Comparativa de configuraciones de transcripción

### 8.1 Variación de parámetros del modelo
- baseline, el mejor que salio en el punto 6
- variante A: cambio un parametro (`beam_size`)
- variante B: cambio un parametro (`condition_on_previous_text`)
- variante C: combinación A + B

### 8.2 Evaluación comparativa (WER)
- comparo las 4 variantes anteriores con WER
- tiempo de ejecucion
- preguntar a GPT si mas métricas

### 8.3 Selección de configuración óptima

Elijo la mejor configuración justificando por qué.

## 9. Selección del *pipeline* final de transcripción

Al igual que en el notebook 1 crear aqui todo el pipeline bien estructurado con sus salidas.

## 10. Implementación y ejecución del *pipeline* final de ASR

### 10.1 Inicialización del entorno y rutas

### Implementación de funciones del *pipeline*

#### 10.2.1 Carga de audios

#### 10.2.2 Transcripción automática

#### 10.2.3 Postprocesado básico del texto

#### 10.2.4 Persistencia de resultados estructurados

### 10.3 Ejecución secuencial del *pipeline*

## 11. Conclusiones
- Resumen del proceso aplicado
- Resultados principales
- Relevancia para el pipeline de Speech-to-Text

In [ ]:
audio_id = "AUDIO-2025-09-18-13-34-03"

item = next(x for x in dataset_without_vad if x["audio_id"] == audio_id)

segments, _ = model.transcribe(str(item["path"]), **asr_params)

text = " ".join([s.text for s in segments]).strip()

print(text)